06/03/2026

Mik va a intentar hacer una red convolucional cn pytorch, lol

Estoy utilizando el env dl2024 

- Cloth**Dataset** guarda la info d vertices x caracteristicas () al acceder a estos items con el **DataLoader** le añade la otra dimension d frames(batchsize) para crear el tensor3D
- Redondear valores para optimizar (ahorra memoria)

Ahora el modelo itera con distintos batchSizes en modo shuffle, para ir entrenandose poco a poco. No tiene memoria, pero como guardamos las velocidades y tal probablemente funcione?
> Your model assumes that the current state of the cloth is all it needs to predict the next state (this is called a Markov assumption). In this setup, the network looks at a single frame's positions and velocities and predicts the displacements. Graph Neural Networks (GNNs) or standard Multi-Layer Perceptrons (MLPs) usually take data in this exact shape.

Otra idea sería:
> When to add a frame dimension (Sequence modeling): If your model needs temporal history—meaning it needs to look at, say, the last 5 frames to figure out what happens in the 6th frame. If you were using an LSTM, RNN, or a Spatiotemporal Transformer, your tensor would need to look like [Batch, Sequence_Length, Vertices, Features].
Por ahora no.

**Links Utilizados:**
- https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html
- https://lixiaoguang.medium.com/build-cnn-from-scratch-5-convolutional-neural-network-86b4d0323fb0

In [6]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import io
import torch
import os, os.path

# WORKING WITH 
datasetPath = 'data/'

def loadAndMergeCSV(csvRoute):
    """
    Carga de los CSV y mergeo en un único CSV. Todos los CSV estarán en la ruta 'data/', y se excluirá el CSV
    'mergedCSV.csv', producto de los mergeos si se ejecutase antes
    """
    csvRoute= 'data/'
    finalData = pd.DataFrame()
    for csvfile in [f for f in os.listdir(csvRoute) if os.path.isfile(csvRoute + f)]:
        if (csvfile != 'mergedCSV.csv' and os.path.splitext(csvfile)[1] == '.csv'):
            data = pd.read_csv(csvRoute + csvfile)
            finalData = pd.concat([data, finalData], ignore_index=True)

    return finalData

cloth_info = loadAndMergeCSV(datasetPath)

print('cloth_info shape: {}'.format(cloth_info.shape))
print('cloth_info: \n{}'.format(cloth_info))

cloth_info shape: (10000, 326)
cloth_info: 
      frame         x0        y0        z0         vx0        vy0        vz0  \
0         0   0.140004 -48.65626  24.86154   -19.50023  2408.4140 -1242.9860   
1         1   0.140004 -48.67581  24.86516   -19.50203  2408.3030 -1243.1080   
2         2   0.140004 -48.66758  24.85419   -19.49655  2408.4560 -1242.7510   
3         3   0.140004 -48.67366  24.86567   -19.50229  2408.7000 -1243.2740   
4         4   0.140004 -48.67369  24.85519   -19.49705  2408.5130 -1242.7420   
...     ...        ...       ...       ...         ...        ...        ...   
9995   4995  19.599910 -46.44315  22.93938  -862.28290  2321.6380 -1072.1200   
9996   4996  36.878800 -41.23395  24.36650 -1699.29400  2101.3170 -1189.2700   
9997   4997  54.104100 -32.39933  24.37123 -2599.23800  1685.9710 -1206.8880   
9998   4998  65.745260 -22.08928  20.64721 -3231.47200  1184.6340 -1037.7880   
9999   4999  68.379470 -17.00436  16.67724 -3430.55100   857.3005  -817.6939

In [7]:
class ClothDataset(Dataset):
    def __init__(self, csv_data, num_vertices=25):
        """
        Args:
            csv_data (str or filepath): Path to the CSV file or raw CSV string.
            num_vertices (int): Number of vertices per frame.
        """
        # Load the CSV data into a pandas DataFrame
        #if isinstance(csv_data, str) and "frame,x0" in csv_data:
            #self.data = pd.read_csv(io.StringIO(csv_data.strip()))
        #else:
            #self.data = pd.read_csv(csv_data)
        self.data = csv_data
            
        #quick fix para espacios en primera fila
        self.data.columns = self.data.columns.str.strip()
        
        self.num_vertices = num_vertices
        
        # Define the base feature names to extract per vertex
        self.feature_prefixes = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'sdf', 'nx', 'ny', 'nz', 'md', 'u', 'v']

        self.position_prefixes = ['x', 'y', 'z']
        self.output_positions = self.data.filter(regex=r'^[xyz]\d+$')

        # Quitamos primera fila de outputs (no es el output de nada) y ultima fila de input (no tiene output)
        self.output_positions = self.output_positions.iloc[1:]
        self.data = self.data.iloc[:-1]
        

    def __len__(self):
        # The number of items is the number of frames (rows) in the dataset
        return len(self.data)
    
    def num_features(self):
        # Number of columns in a row (pos, vel, sdf, uv per vertex)
        return len(self.feature_prefixes)
    
    def num_vertex(self):
        return self.num_vertices
    
    def _get_frame_output_tensor(self, idx):
         row = self.output_positions.iloc[idx]
         frame_data = []
         for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.position_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

         tensor_data = torch.tensor(np.array(frame_data))

         return tensor_data
    
    def _get_frame_tensor(self, idx):
        row = self.data.iloc[idx]
        frame_data = []

        for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.feature_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

        tensor_data = torch.tensor(np.array(frame_data))

        return tensor_data
        
    def __getitem__(self, idx):
        frame_t = self._get_frame_tensor(idx)
        frame_t1 = self._get_frame_output_tensor(idx) # +1 ya no TODO: Pillar solo las columnas de pos

        return frame_t, frame_t1

# --- Example Usage ---

# (Assuming 'csv_string' is a variable holding your provided data block)
dataset = ClothDataset(cloth_info)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

for batch_data, batch_frames in dataloader:
    print(f"Batch Shape: {batch_data.shape}")
    print(batch_data)
    print(batch_frames)
    break


mean = 0
std = 0
n_samples = 0

for batch_t, _ in dataloader:
    batch_t = batch_t.float()
    
    batch_samples = batch_t.size(0)
    batch_t = batch_t.view(-1, batch_t.size(-1))  

    mean += batch_t.mean(dim=0)
    std += batch_t.std(dim=0)
    n_samples += 1

mean /= n_samples
std /= n_samples

# evitamos division por 0
std[std < 1e-8] = 1.0

print("MEAN:", mean)
print("STD:", std)

Batch Shape: torch.Size([4, 25, 13])
tensor([[[ 58.3627, -29.8582,  23.9595,  ...,   1.0000,   0.7500,   0.0000],
         [ 44.7350,  -8.9864,  49.0141,  ...,   1.0000,   1.0000,   0.2500],
         [ 62.4833, -26.7826,  48.3373,  ...,   1.0000,   1.0000,   0.0000],
         ...,
         [  8.5664,  27.8080, -49.4666,  ...,   1.0000,   0.0000,   0.7500],
         [  0.1400,  51.3300, -25.1385,  ...,   0.0000,   0.2500,   1.0000],
         [  0.1400,  51.3300, -50.1385,  ...,   0.0000,   0.0000,   1.0000]],

        [[ 42.8353, -38.8539,  23.9493,  ...,   1.0000,   0.7500,   0.0000],
         [ 39.9215, -11.8883,  46.3180,  ...,   1.0000,   1.0000,   0.2500],
         [ 57.6496, -29.1493,  41.5969,  ...,   1.0000,   1.0000,   0.0000],
         ...,
         [ 10.2646,  28.4756, -49.8008,  ...,   1.0000,   0.0000,   0.7500],
         [  0.1400,  51.3300, -25.1385,  ...,   0.0000,   0.2500,   1.0000],
         [  0.1400,  51.3300, -50.1385,  ...,   0.0000,   0.0000,   1.0000]],

       

In [1]:
import json
# Intento de normalización uep

# Convertir tensores a listas
norm_data = {
    "mean": mean.tolist(),
    "std": std.tolist(),
    "feature_prefixes": ['x', 'y', 'z', 'vx', 'vy', 'vz', 'sdf', 'nx', 'ny', 'nz', 'md', 'u', 'v']
}

with open("cloth_norm_params.json", "w") as f:
    json.dump(norm_data, f)

NameError: name 'mean' is not defined

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ClothPhysicsNet(nn.Module):
    def __init__(self, num_inputs=13, hidden_dim=128, num_outputs=3):
        super().__init__()
        # Wider and deeper network
        self.fc_in = nn.Linear(num_inputs, hidden_dim)
        
        # Hidden layers for complex physics calculations
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        
        self.fc_out = nn.Linear(hidden_dim, num_outputs)

    def forward(self, x):
        # Using ELU instead of ReLU for smoother gradients in physics
        x = F.elu(self.fc_in(x))
        
        # Residual Block 1
        res = x
        x = F.elu(self.fc1(x))
        x = F.elu(self.fc2(x))
        x = x + res # Skip connection helps stability in deeper networks
        
        x = F.elu(self.fc3(x))
        x = self.fc_out(x)
        
        return x

# --- PRE-COMPUTED DATASET STATS (DO NOT calculate per batch) ---
# You must calculate these across your WHOLE dataset before training!
# Example placeholder tensors:
global_input_mean = torch.zeros(13) 
global_input_std = torch.ones(13)
global_delta_mean = torch.zeros(3)
global_delta_std = torch.ones(3)

model = ClothPhysicsNet(num_inputs=13, hidden_dim=128, num_outputs=3)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(100):
    for batch_t, batch_t1 in dataloader:
        batch_t = batch_t.float()
        batch_t1 = batch_t1.float()

        # 1. Normalize inputs using GLOBAL stats
        batch_t_norm = (batch_t - global_input_mean) / global_input_std

        # 2. Calculate the DELTA (displacement) instead of absolute position
        # Assuming the first 3 indices of batch_t are position x,y,z
        positions_t = batch_t[..., 0:3] 
        target_delta = batch_t1 - positions_t

        # 3. Normalize the target DELTA using GLOBAL stats
        target_delta_norm = (target_delta - global_delta_mean) / global_delta_std

        # Forward pass
        pred_delta_norm = model(batch_t_norm)

        # Loss based on the normalized displacement
        loss = criterion(pred_delta_norm, target_delta_norm)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

Epoch 1, Loss: 24.101055145263672
Epoch 2, Loss: 54.060855865478516
Epoch 3, Loss: 53.03121566772461
Epoch 4, Loss: 57.302223205566406
Epoch 5, Loss: 47.97121810913086
Epoch 6, Loss: 61.183555603027344
Epoch 7, Loss: 44.003944396972656
Epoch 8, Loss: 7.777525901794434
Epoch 9, Loss: 19.32188606262207
Epoch 10, Loss: 14.196276664733887
Epoch 11, Loss: 17.016353607177734
Epoch 12, Loss: 28.043174743652344
Epoch 13, Loss: 11.670970916748047
Epoch 14, Loss: 44.0244255065918
Epoch 15, Loss: 7.824741363525391
Epoch 16, Loss: 14.78575611114502
Epoch 17, Loss: 13.391426086425781
Epoch 18, Loss: 6.104640960693359
Epoch 19, Loss: 8.933826446533203
Epoch 20, Loss: 14.394533157348633
Epoch 21, Loss: 4.281073570251465
Epoch 22, Loss: 4.993649005889893
Epoch 23, Loss: 23.246803283691406
Epoch 24, Loss: 4.709444046020508
Epoch 25, Loss: 4.063549041748047
Epoch 26, Loss: 9.497762680053711
Epoch 27, Loss: 6.974025726318359
Epoch 28, Loss: 2.7563018798828125
Epoch 29, Loss: 4.846622943878174
Epoch 30, L

In [4]:
# TODO
# una recurrente sencilla (la salida se vuelve entrada en el siguiente ejemplo)
# Antes de meternos en CNN y LSTM
# AÑADIMOS VALORES U V PARA CADA VERTICE ( no queremos perder la noción espacial )

import torch.nn as nn
import torch.nn.functional as F #acceso rapido a funciones
import torch.utils.data as data #cargar y manejar el training data

class MyModule(nn.Module):

    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()
        # Some init for my module
        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Function for performing the calculation of the module.
        
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        #print(x)
        return x

    #backward se hace automaticamente, podriamos definirla tmbn 

#tmbn clases DataSet y DataLoader

# definir modelo, loss function y optimizer
#TODO: buscar dimensiones reales de las neuronas

# nn.Linear in PyTorch is designed to handle 3D tensors seamlessly.  
# When a 3D input tensor (e.g., batch_size, sequence_length, features) is provided,
#  the layer applies the linear transformation only to the last dimension (the features dimension),
#  preserving all other dimensions.

model = MyModule(num_inputs=13, num_hidden= 64, num_outputs=3)
# print, save, lo que sea
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(50):
    for batch_t, batch_t1 in dataloader:

        batch_t = batch_t.float()
        batch_t1 = batch_t1.float()

        batch_t = (batch_t - mean) / std

        # output (posiciones)
        mean_out = batch_t1.mean(dim=(0,1), keepdim=True)
        std_out = batch_t1.std(dim=(0,1), keepdim=True) + 1e-8
        batch_t1 = (batch_t1 - mean_out) / std_out

        pred = model(batch_t)

        loss = criterion(pred, batch_t1)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        print(f'Epoch {epoch+1}, Loss: {loss.item()}')


Epoch 1, Loss: 1.0472075939178467
Epoch 1, Loss: 1.0464675426483154
Epoch 1, Loss: 1.0303698778152466
Epoch 1, Loss: 1.0499141216278076
Epoch 1, Loss: 1.0316685438156128
Epoch 1, Loss: 1.0490152835845947
Epoch 1, Loss: 1.0630629062652588
Epoch 1, Loss: 1.0475143194198608
Epoch 1, Loss: 1.0150617361068726
Epoch 1, Loss: 1.0244077444076538
Epoch 1, Loss: 1.0253373384475708
Epoch 1, Loss: 1.0314913988113403
Epoch 1, Loss: 1.0160117149353027
Epoch 1, Loss: 1.0378696918487549
Epoch 1, Loss: 1.007398009300232
Epoch 1, Loss: 1.0114015340805054
Epoch 1, Loss: 0.9971711039543152
Epoch 1, Loss: 0.9958064556121826
Epoch 1, Loss: 1.0196338891983032
Epoch 1, Loss: 1.008742332458496
Epoch 1, Loss: 0.9922777414321899
Epoch 1, Loss: 1.002906084060669
Epoch 1, Loss: 0.9804728031158447
Epoch 1, Loss: 0.9798946380615234
Epoch 1, Loss: 0.9789349436759949
Epoch 1, Loss: 0.9619595408439636
Epoch 1, Loss: 0.9751774072647095
Epoch 1, Loss: 0.9919008612632751
Epoch 1, Loss: 0.9686527252197266
Epoch 1, Loss: 0.

In [9]:
#INTENTO DE EXPORTAR A ONNX
import sys
print(sys.executable)

import onnx
import onnxruntime

print("ONNX version:", onnx.__version__)
print("ONNX Runtime version:", onnxruntime.__version__)

# Create example inputs for exporting the model. The inputs should be a tuple of tensors.
example_inputs = (batch_t)
onnx_program = torch.onnx.export(model, example_inputs, dynamo=True)

onnx_program.save("onnxModels/trainedLinearBigMovModel.onnx")

c:\Users\mikel\miniconda3\envs\dl2024\python.exe
ONNX version: 1.21.0
ONNX Runtime version: 1.24.4
[torch.onnx] Obtain model graph for `ClothPhysicsNet([...]` with `torch.export.export`...
[torch.onnx] Obtain model graph for `ClothPhysicsNet([...]` with `torch.export.export`... ✅
[torch.onnx] Translate the graph into ONNX...


W0420 02:04:16.479000 20884 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0420 02:04:16.479000 20884 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0420 02:04:16.485000 20884 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0420 02:04:16.487000 20884 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 

[torch.onnx] Translate the graph into ONNX... ✅


In [6]:
import torch
import torch.nn as nn

#EJEMPLO SENCILLO CONVOLUCIONAL PARA MÁS ADELANTE

# Example: 100 features, 1 channel (linear input)
# Batch size=16
input_data = torch.randn(16, 1, 100) 

model = nn.Sequential(
    nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3), # Extract features
    nn.ReLU(),
    nn.Flatten(), # Flatten for Dense layer
    nn.Linear(32 * 98, 10) # 98 is the new length after convolution
)
